In [ ]:
# Table S13: observed split-to-split coverage of class-conditional conformal prediction at the 90% target,
# compared with the variation expected under exchangeability for the calibration and test sizes of each split.
# Add as a cell in 4_supplementary.ipynb (after Table S8). Reads modeling_table.csv and, from core_outputs/ (1_code.ipynb):
#   conformal_200_splits.csv, is_test_egd.npy, is_test_utsw.npy. Writes core_outputs/tableS13_expected_variation.csv
#
# Expected variation: with n calibration patients of a subtype and a target of 90%, the cutoff is the
# k-th smallest calibration score, k = ceil(0.9 * (n + 1)). Under exchangeability, the coverage of a future patient
# given that calibration group follows Beta(k, n + 1 - k). The number of covered patients among m test patients
# of that subtype then follows a beta-binomial distribution with parameters (m, k, n + 1 - k).
# Each split has its own n and m, so the expected distribution for a subtype is the average over the 200 splits.

import os
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import betabinom

OUT = 'core_outputs'
target = 0.90
table = pd.read_csv('modeling_table.csv', usecols=['site', '2021_glioma_class'])
classes = np.array(['astrocytoma', 'glioblastoma', 'oligodendroglioma'])
y = {'EGD': table[table['site'] == 'EGD']['2021_glioma_class'].values,
     'UTSW': table[table['site'] == 'UTSW']['2021_glioma_class'].values}
is_test = {'EGD': np.load(os.path.join(OUT, 'is_test_egd.npy')), 'UTSW': np.load(os.path.join(OUT, 'is_test_utsw.npy'))}

# ---- 1. calibration size n and test size m of each subtype in each split ----
size_rows = []
for cohort, seed, cls in product(['EGD', 'UTSW'], range(200), classes):
    in_class = y[cohort] == cls
    m = int(np.sum(in_class & is_test[cohort][seed]))
    n = int(np.sum(in_class & ~is_test[cohort][seed]))
    size_rows.append({'cohort': cohort, 'seed': seed, 'class': cls, 'n_cal': n, 'm_test': m})
sizes = pd.DataFrame(size_rows)

# ---- 2. observed coverage per split at 90% (from 1_code.ipynb), with the covered count ----
cp = pd.read_csv(os.path.join(OUT, 'conformal_200_splits.csv'))
cp = cp[np.isclose(cp['target'], target)]
cp = cp.merge(sizes, on=['cohort', 'seed', 'class'])
cp['covered_count'] = np.round(cp['covered'] * cp['m_test']).astype(int)
cp['needed'] = -(-9 * cp['m_test'] // 10)                 # covered count needed to reach 90% = ceil(0.9 * m), integer arithmetic
cp['below'] = cp['covered_count'] < cp['needed']

# ---- 3. expected distribution for each split: beta-binomial(m, k, n + 1 - k) ----
cp['k'] = -(-9 * (cp['n_cal'] + 1) // 10)                  # ceil(0.9 * (n + 1)), integer arithmetic

cp['b'] = cp['n_cal'] + 1 - cp['k']
cp['p_below'] = betabinom.cdf(cp['needed'] - 1, cp['m_test'], cp['k'], cp['b'])   # chance this split falls below 90%

# probability of every possible covered count (0 to largest m) for every split, one row per split
m_max = int(cp['m_test'].max())
x = np.arange(m_max + 1)
pmf = betabinom.pmf(x[None, :], cp['m_test'].values[:, None], cp['k'].values[:, None], cp['b'].values[:, None])
coverage_value = x[None, :] / cp['m_test'].values[:, None]      # covered count as a coverage fraction
pmf[x[None, :] > cp['m_test'].values[:, None]] = 0              # counts above m are impossible

# ---- 4. summary per cohort and subtype ----
s13_rows = []
for cohort, cls in product(['EGD', 'UTSW'], classes):
    rows = np.where((cp['cohort'] == cohort).values & (cp['class'] == cls).values)[0]
    d = cp.iloc[rows]

    # expected 2.5th percentile: pool the 200 split distributions with equal weight, then read the 2.5th percentile
    values = coverage_value[rows].ravel()
    weights = pmf[rows].ravel() / len(rows)
    order = np.argsort(values)
    cumulative = np.cumsum(weights[order])
    expected_p025 = values[order][np.searchsorted(cumulative, 0.025)]

    s13_rows.append({'Cohort': 'EGD' if cohort == 'EGD' else 'UTSW-Glioma',
                     'Subtype': cls.capitalize(),
                     'Calibration patients': '-'.join(str(v) for v in sorted(d['n_cal'].unique())),
                     'Test patients': '-'.join(str(v) for v in sorted(d['m_test'].unique())),
                     'Splits below 90%, observed (%)': round(d['below'].mean() * 100, 1),
                     'Splits below 90%, expected (%)': round(d['p_below'].mean() * 100, 1),
                     'Mean coverage, observed': round(d['covered'].mean(), 2),
                     '2.5th percentile, observed': round(d['covered'].quantile(0.025), 2),
                     '2.5th percentile, expected': round(expected_p025, 2)})

tableS13 = pd.DataFrame(s13_rows)
tableS13.to_csv(os.path.join(OUT, 'tableS13_expected_variation.csv'), index=False)
print(tableS13.to_string(index=False))

     Cohort           Subtype Calibration patients Test patients  Splits below 90%, observed (%)  Splits below 90%, expected (%)  Mean coverage, observed  2.5th percentile, observed  2.5th percentile, expected
        EGD       Astrocytoma                38-39         38-39                            31.0                            42.1                     0.93                        0.79                        0.76
        EGD      Glioblastoma                  127           127                            38.5                            41.5                     0.91                        0.84                        0.83
        EGD Oligodendroglioma                36-37         36-37                            43.0                            33.8                     0.91                        0.78                        0.78
UTSW-Glioma       Astrocytoma                   38            39                            31.0                            35.0                     0.93       